# Test Recommendation Models

Notebook này dùng để test các model recommendation đã train (UserCF, ItemCF, SVD, Content-Based).

Người dùng có thể:
1. Chọn một user hoặc nhập user ID
2. Chọn model recommendation (UserCF, ItemCF, SVD, Content-Based, hoặc All)
3. Nhận top-K gợi ý phim, dự đoán rating, và thông tin chi tiết

**4 mô hình có sẵn:**
- **UserCF**: Tìm users tương tự, dự đoán dựa trên rating của users tương tự
- **ItemCF**: Tìm items tương tự, dự đoán dựa trên rating của items tương tự
- **SVD**: Matrix Factorization, dự đoán qua latent factors
- **Content-Based**: Dựa vào đặc trưng phim, giải quyết cold-start problem

## 1. Tải các model đã train

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.metrics.pairwise import cosine_similarity

# Tải metadata
print("Loading models...")
metadata = joblib.load('models/model_metadata.pkl')
usercf_model = joblib.load('models/usercf_model.pkl')
itemcf_model = joblib.load('models/itemcf_model.pkl')
svd_model = joblib.load('models/svd_model.pkl')

try:
    content_model = joblib.load('models/content_model.pkl')
    print("✓ Content-Based model loaded!")
except Exception as e:
    print(f"Content-Based model not found: {e}")
    content_model = None

# Load df_train từ CSV (lưu riêng để tối ưu memory)
df_train = pd.read_csv('models/df_train.csv')
print(f"✓ Training data loaded: {len(df_train)} ratings")

df_movies = pd.read_csv('models/movies_metadata_for_testing.csv')

print("✓ All models loaded successfully!")
print(f"\nAvailable {len(metadata['user_ids'])} users and {len(metadata['movie_ids'])} movies")

# Lấy các thông tin cần thiết
user_to_idx = metadata['user_to_idx']
idx_to_user = metadata['idx_to_user']
movie_to_idx = metadata['movie_to_idx']
idx_to_movie = metadata['idx_to_movie']
user_ids = metadata['user_ids']
movie_ids = metadata['movie_ids']

Loading models...
✓ Content-Based model loaded!
✓ Training data loaded: 100224 ratings
✓ All models loaded successfully!

Available 610 users and 9697 movies


## 2. Định nghĩa hàm predict cho từng model

In [ ]:
def predict_usercf(user_id, movie_id, k=20):
    """Predict rating using UserCF"""
    user_sim = usercf_model['user_sim']
    R = usercf_model['R']
    # user_to_idx e movie_to_idx được load từ metadata trong Section 1 (global variables)
    
    if user_id not in user_to_idx or movie_id not in movie_to_idx:
        return np.nan
    
    uidx = user_to_idx[user_id]
    midx = movie_to_idx[movie_id]
    sims = user_sim[uidx]
    ratings = R[:, midx].toarray().reshape(-1)
    
    mask = ratings > 0
    if mask.sum() == 0:
        return np.nan
    
    sims = sims[mask]
    ratings = ratings[mask]
    top_k = np.argsort(sims)[-k:]
    sim_k = sims[top_k]
    rating_k = ratings[top_k]
    
    if sim_k.sum() == 0:
        return np.nan
    
    return np.dot(sim_k, rating_k) / sim_k.sum()

def predict_itemcf(user_id, movie_id, k=20):
    """Predict rating using ItemCF"""
    item_sim = itemcf_model['item_sim']
    R = itemcf_model['R']
    # user_to_idx e movie_to_idx được load từ metadata trong Section 1 (global variables)
    
    if user_id not in user_to_idx or movie_id not in movie_to_idx:
        return np.nan
    
    uidx = user_to_idx[user_id]
    midx = movie_to_idx[movie_id]
    user_ratings = R[uidx].toarray().reshape(-1)
    mask = user_ratings > 0
    
    if mask.sum() == 0:
        return np.nan
    
    sims = item_sim[midx, mask]
    rating_k = user_ratings[mask]
    top_k = np.argsort(sims)[-k:]
    sim_k = sims[top_k]
    rating_k = rating_k[top_k]
    
    if sim_k.sum() == 0:
        return np.nan
    
    return np.dot(sim_k, rating_k) / sim_k.sum()

def predict_svd(user_id, movie_id):
    """Predict rating using Surprise SVD"""
    # no manual index mapping needed, just predict using user_id and movie_id
    try:
        return svd_model.predict(user_id, movie_id).est
    except Exception:
        return np.nan
def predict_contentbased(user_id, movie_id):
    """
    Predict rating using Content-Based Recommendation
    - User profile = trung bình vector đặc trưng của phim đã rate
    - Tính cosine similarity giữa user profile và movie feature vector
    - Scale kết quả về range [0, 5]
    - Hỗ trợ sparse matrix format (tiết kiệm RAM)
    """
    if content_model is None:
        return np.nan
    
    user_profile = content_model.get('user_profile', {})
    movie_vecs = content_model.get('movie_vec', {})
    
    if user_id not in user_profile or movie_id not in movie_vecs:
        return np.nan
    
    profile = user_profile[user_id]
    if profile is None:
        return np.nan
    
    # movie_vecs[movie_id] có thể là sparse CSR matrix hoặc dense numpy array
    # cosine_similarity xử lý cả hai format
    movie_vec = movie_vecs[movie_id]
    
    # Nếu profile là sparse, convert thành sparse; nếu dense thì reshape
    if hasattr(profile, 'todense'):  # Profile là sparse
        sim = cosine_similarity(profile.reshape(1, -1), movie_vec)[0, 0]
    elif hasattr(movie_vec, 'todense'):  # Movie vec là sparse
        sim = cosine_similarity(profile.reshape(1, -1), movie_vec)[0, 0]
    else:  # Cả hai đều dense
        sim = cosine_similarity(profile.reshape(1, -1), movie_vec.reshape(1, -1))[0, 0]
    
    return np.clip(sim * 5.0, 0.0, 5.0)

print("✓ Prediction functions defined")

✓ Prediction functions defined


## 3. Hàm gợi ý top-K phim cho một user

In [3]:
def recommend_movies(user_id, k=10, model_type='itemcf', threshold=3.5):
    """
    Gợi ý top-K phim cho user sử dụng model chỉ định
    
    Parameters:
    - user_id: ID của user
    - k: Số phim gợi ý
    - model_type: 'usercf', 'itemcf', 'svd', hoặc 'content'
    - threshold: Chỉ gợi ý phim có predicted rating >= threshold
    """
    
    if user_id not in user_to_idx:
        return None, f"User {user_id} not found in training data"
    
    # Lấy danh sách phim đã rated của user từ training data (df_train được load từ models/df_train.csv)
    user_rated_movies = set(df_train[df_train['userId'] == user_id]['movieId'].values)
    
    # Chọn hàm predict
    if model_type.lower() == 'usercf':
        predict_fn = predict_usercf
        model_desc = "User-based Collaborative Filtering"
    elif model_type.lower() == 'itemcf':
        predict_fn = predict_itemcf
        model_desc = "Item-based Collaborative Filtering"
    elif model_type.lower() == 'svd':
        predict_fn = predict_svd
        model_desc = "Matrix Factorization (SVD)"
    elif model_type.lower() in ['content', 'content-based', 'contentbased']:
        if content_model is None:
            return None, "Content-Based model not loaded!"
        predict_fn = predict_contentbased
        model_desc = "Content-Based Recommendation"
    else:
        return None, f"Unknown model type: {model_type}. Use: usercf, itemcf, svd, or content"
    
    # Tính score cho tất cả phim chưa rated
    scores = []
    for movie_id in movie_ids:
        if movie_id in user_rated_movies:
            continue  # Bỏ qua phim đã rated
        
        score = predict_fn(user_id, movie_id)
        if not np.isnan(score) and score >= threshold:
            scores.append((movie_id, score))
    
    # Sort và lấy top-k
    scores.sort(key=lambda x: x[1], reverse=True)
    top_movies = scores[:k]
    
    # Tạo DataFrame kết quả
    results = []
    for idx, (movie_id, score) in enumerate(top_movies, 1):
        movie_info = df_movies[df_movies['movieId'] == movie_id].iloc[0]
        results.append({
            'rank': idx,
            'movieId': movie_id,
            'title': movie_info.get('title', 'N/A'),
            'genres': movie_info.get('genres', 'N/A'),
            'predicted_rating': score,
            'predicted_score': f"{score:.2f}/5.0"
        })
    
    return pd.DataFrame(results), None

print("✓ Recommendation function defined (supports all 4 models)")

✓ Recommendation function defined (supports all 4 models)


## 4. Hiển thị thông tin model

In [ ]:
print("="*70)
print("MODEL PERFORMANCE SUMMARY")
print("="*70)

print("\nRMSE (Root Mean Square Error) - Độ lệch căn bậc hai:")
print(f"  UserCF:  {metadata['rmse_scores'].get('usercf', 0):.4f}")
print(f"  ItemCF:  {metadata['rmse_scores'].get('itemcf', 0):.4f} ← BEST RMSE")
print(f"  SVD:     {metadata['rmse_scores'].get('svd', 0):.4f}")
print(f"  Content: {metadata['rmse_scores'].get('content', 0):.4f}")

print("\nMAE (Mean Absolute Error) - Sai số tuyệt đối trung bình:")
print(f"  UserCF:  {metadata['mae_scores'].get('usercf', 0):.4f}")
print(f"  ItemCF:  {metadata['mae_scores'].get('itemcf', 0):.4f} ← BEST MAE")
print(f"  SVD:     {metadata['mae_scores'].get('svd', 0):.4f}")
print(f"  Content: {metadata['mae_scores'].get('content', 0):.4f}")

print("\nPrecision@10 - Độ chính xác gợi ý (% phim gợi ý đúng):")
print(f"  UserCF:  {metadata['precision_at_10'].get('usercf', 0):.4f}")
print(f"  ItemCF:  {metadata['precision_at_10'].get('itemcf', 0):.4f}")
print(f"  SVD:     {metadata['precision_at_10'].get('svd', 0):.4f} ← BEST Precision")
print(f"  Content: {metadata['precision_at_10'].get('content', 0):.4f}")

print("\nRecall@10 - Tỷ lệ phim đúng được tìm thấy:")
print(f"  UserCF:  {metadata['recall_at_10'].get('usercf', 0):.4f}")
print(f"  ItemCF:  {metadata['recall_at_10'].get('itemcf', 0):.4f}")
print(f"  SVD:     {metadata['recall_at_10'].get('svd', 0):.4f} ← BEST Recall")
print(f"  Content: {metadata['recall_at_10'].get('content', 0):.4f}")

MODEL PERFORMANCE SUMMARY

RMSE (Root Mean Square Error) - Độ lệch căn bậc hai:
  UserCF:  1.0473
  ItemCF:  0.9499 ← BEST RMSE
  SVD:     0.9651
  Content: 1.7209

MAE (Mean Absolute Error) - Sai số tuyệt đối trung bình:
  UserCF:  0.8281
  ItemCF:  0.7146 ← BEST MAE
  SVD:     0.7422
  Content: 1.3213

Precision@10 - Độ chính xác gợi ý (% phim gợi ý đúng):
  UserCF:  0.0000
  ItemCF:  0.0002
  SVD:     0.0007 ← BEST Precision
  Content: 0.0000

Recall@10 - Tỷ lệ phim đúng được tìm thấy:
  UserCF:  0.0000
  ItemCF:  0.0016
  SVD:     0.0066 ← BEST Recall
  Content: 0.0000

KẾT LUẬN VÀ KHUYẾN NGHỊ:

📊 Theo Prediction Accuracy (RMSE/MAE):
   ItemCF - SAI SỐ DỰ ĐOÁN THẤP NHẤT
   → Dùng khi: Cần dự đoán rating chính xác
   → Ưu điểm: Stable, dễ triển khai, nhanh

🎯 Theo Ranking Quality (Precision/Recall):
   SVD - XẾP HẠNG PHIM TỐT NHẤT
   → Dùng khi: Muốn gợi ý phim liên quan chất lượng cao
   → Ưu điểm: Học latent factors, xử lý sparsity tốt

🆕 Cho Cold-Start Problem (người dùng/phim mớ

## 5. Test gợi ý cho một user cụ thể

In [ ]:
# Chọn user để test
test_user_id = 1  # Có thể thay đổi thành user_id khác

print(f"\n{'='*70}")
print(f"TEST RECOMMENDATIONS FOR USER: {test_user_id}")
print(f"{'='*70}")

# Lấy thông tin user từ df_train (đã được load trong section 1 từ models/df_train.csv)
user_ratings = df_train[df_train['userId'] == test_user_id]
print(f"\nUser {test_user_id} đã rate {len(user_ratings)} phim")
print(f"Average rating: {user_ratings['rating'].mean():.2f}/5.0")

print(f"\nSome movies rated by user {test_user_id}:")
sample_rated = user_ratings.head(3)
for _, row in sample_rated.iterrows():
    movie_info = df_movies[df_movies['movieId'] == row['movieId']].iloc[0]
    print(f"  • {movie_info['title']} ({movie_info['genres']}) - Rating: {row['rating']}/5")

# Test các model
print(f"\n{'-'*70}")
print(f"RECOMMENDATIONS USING DIFFERENT MODELS")
print(f"{'-'*70}")

for model_name in ['itemcf', 'svd', 'usercf', 'content']:
    print(f"\n>>> Model: {model_name.upper()}")
    if model_name == 'content' and content_model is None:
        print(f"Content-Based model not loaded!")
        continue
    df_recs, error = recommend_movies(test_user_id, k=5, model_type=model_name, threshold=2.5)
    
    if error:
        print(f"  Error: {error}")
    else:
        print(f"\nTop 5 Recommendations:")
        print(df_recs.to_string(index=False))


TEST RECOMMENDATIONS FOR USER: 1

User 1 đã rate 231 phim
Average rating: 4.37/5.0

Some movies rated by user 1:
  • Toy Story (1995) (['Adventure' 'Animation' 'Children' 'Comedy' 'Fantasy']) - Rating: 4.0/5
  • Grumpier Old Men (1995) (['Comedy' 'Romance']) - Rating: 4.0/5
  • Heat (1995) (['Action' 'Crime' 'Thriller']) - Rating: 4.0/5

----------------------------------------------------------------------
RECOMMENDATIONS USING DIFFERENT MODELS
----------------------------------------------------------------------

>>> Model: ITEMCF

Top 5 Recommendations:
 rank  movieId                                                title                                     genres  predicted_rating predicted_score
    1     1140    Entertaining Angels: The Dorothy Day Story (1996)                                  ['Drama']               5.0        5.00/5.0
    2     1519                                Broken English (1996)                                  ['Drama']               5.0        5.00/5.0


## 6. Interactive Testing - Thử với user khác

In [6]:
# Ví dụ: Test với user có rating nhiều nhất
user_activity = df_train.groupby('userId').size()
most_active_user = user_activity.idxmax()

print(f"Most active user: {most_active_user} (rated {user_activity.max()} movies)")
print(f"\nTop 10 recommendations for user {most_active_user} using ItemCF:")

df_recs, error = recommend_movies(most_active_user, k=10, model_type='itemcf')
if error:
    print(f"Error: {error}")
else:
    print(df_recs.to_string(index=False))

Most active user: 414 (rated 2697 movies)

Top 10 recommendations for user 414 using ItemCF:
 rank  movieId                                                      title                                  genres  predicted_rating predicted_score
    1     1219                                              Psycho (1960)                      ['Crime' 'Horror']          4.902959        4.90/5.0
    2     1203                                        12 Angry Men (1957)                               ['Drama']          4.701834        4.70/5.0
    3     1258                                        Shining, The (1980)                              ['Horror']          4.701532        4.70/5.0
    4     3681 For a Few Dollars More (Per qualche dollaro in più) (1965) ['Action' 'Drama' 'Thriller' 'Western']          4.664077        4.66/5.0
    5     3462                                        Modern Times (1936)            ['Comedy' 'Drama' 'Romance']          4.601054        4.60/5.0
    6   189043     

## 7. Summary & Next Steps

### Summary
- 4 models have been trained and saved (UserCF, ItemCF, SVD, Content-Based)
- ItemCF provides best prediction accuracy (RMSE, MAE)
- SVD provides best ranking quality (Precision, Recall)
- Content-Based handles cold-start problem well
- All models can predict ratings and recommend movies

### How to Use Each Model
1. **ItemCF** - Best for accurate rating prediction
   - `recommend_movies(user_id, k=10, model_type='itemcf')`
   
2. **SVD** - Best for top-K recommendations
   - `recommend_movies(user_id, k=10, model_type='svd')`
   
3. **Content-Based** - Best for new users/items
   - `recommend_movies(user_id, k=10, model_type='content')`
   
4. **UserCF** - Best for finding similar users
   - `recommend_movies(user_id, k=10, model_type='usercf')`

### Common Parameters
- `k` - Number of recommendations (default: 10)
- `threshold` - Minimum predicted rating to include (default: 2.5)
- `model_type` - 'usercf', 'itemcf', 'svd', or 'content'